In [2]:
#pwd

In [3]:
import pandas as pd
import os


In [ ]:

BASE = os.path.join(os.getcwd(), "..")  # project root

customers   = pd.read_csv(os.path.join(BASE, "Data", "raw", "customers.csv"))
orders      = pd.read_csv(os.path.join(BASE, "Data", "raw", "orders.csv"))
order_items = pd.read_csv(os.path.join(BASE, "Data", "raw", "order_items.csv"))
shipments   = pd.read_csv(os.path.join(BASE, "Data", "raw", "shipments.csv"))

In [5]:
# CUSTOMER TABLE - DATA QUALITY & EXPLORATION CHECKS

print(f"columns present in customer's data are : \n{" ,".join(list(customers.columns))}")



columns present in customer's data are : 
customer_id ,signup_date ,city ,state ,segment


In [6]:
# 1. DATA TYPES
print("=== Data Types ===")
print(customers.dtypes)

# Fix: signup_date comes as string → convert to datetime
# normalize() removes the time component (00:00:00) and keeps it as pure date
customers["signup_date"] = pd.to_datetime(customers["signup_date"]).dt.normalize()


=== Data Types ===
customer_id    str
signup_date    str
city           str
state          str
segment        str
dtype: object


In [7]:
# 2. DUPLICATE CHECK
# customers is a primary table → should have no duplicate customer_ids

dup_count = customers.duplicated(subset="customer_id").sum()
print(f"\n Duplicate customer_id count: {dup_count}")



 Duplicate customer_id count: 0


In [8]:
# 3. MISSING VALUES
missing_val = customers.isna().sum()
print("\nMissing Values")
print(missing_val)


Missing Values
customer_id    0
signup_date    0
city           0
state          0
segment        0
dtype: int64


In [9]:
# 4. CATEGORICAL COLUMNS - CONSISTENCY CHECK
# Check for inconsistencies like "Bangalore" vs "bangalore" vs "bangaLore"
print("\nUnique Cities")
unique_city  = sorted(customers["city"].unique().tolist())
print(unique_city)



Unique Cities
['Ahmedabad', 'Bangalore', 'Chandigarh', 'Chennai', 'Delhi', 'Hyderabad', 'Jaipur', 'Kochi', 'Kolkata', 'Lucknow', 'Mumbai', 'Pune']


In [10]:
print("\nUnique States")
unique_state = sorted(customers["state"].unique().tolist())
print(unique_state)



Unique States
['Delhi', 'Gujarat', 'Karnataka', 'Kerala', 'Maharashtra', 'Punjab', 'Rajasthan', 'Tamil Nadu', 'Telangana', 'Uttar Pradesh', 'West Bengal']


In [11]:
print("\nUnique Segments")
unique_seg   = sorted(customers["segment"].unique().tolist())
print(unique_seg)


Unique Segments
['Budget', 'Premium', 'Value']


In [12]:
# 5. COLUMN SUMMARY
cat_cols         = customers.select_dtypes(include=["str"]).columns.tolist()
num_or_date_cols = customers.select_dtypes(exclude=["str"]).columns.tolist()
print(f"\nCategorical Columns  : {', '.join(cat_cols)}")
print(f"Numeric/Date Columns : {', '.join(num_or_date_cols)}")


Categorical Columns  : customer_id, city, state, segment
Numeric/Date Columns : signup_date


In [13]:
# 6. KEY STATS SUMMARY
total_customers = customers["customer_id"].nunique()
min_signup_date = customers["signup_date"].min().strftime("%Y-%m-%d")
max_signup_date = customers["signup_date"].max().strftime("%Y-%m-%d")

In [14]:
print(f"""
========================================
        CUSTOMER TABLE SUMMARY
========================================
  Total Customers   : {total_customers}
  Signup Date Range : {min_signup_date} → {max_signup_date}
  Cities            : {len(unique_city)}
  States            : {len(unique_state)}
  Segments          : {unique_seg}
  Duplicates        : {dup_count}
  Missing Values    : {missing_val.any()}
========================================
""")


        CUSTOMER TABLE SUMMARY
  Total Customers   : 25000
  Signup Date Range : 2022-07-03 → 2024-07-01
  Cities            : 12
  States            : 11
  Segments          : ['Budget', 'Premium', 'Value']
  Duplicates        : 0
  Missing Values    : False



Understanding customer's behaviour

In [15]:
# 1. What is the monthly/yearly signup trend? 
# Are we growing or slowing down in customer acquisition over time?
customers["signup_year"] = customers["signup_date"].dt.year
customers["signup_month_num"] = customers["signup_date"].dt.month
customers["signup_month"] = customers["signup_date"].dt.month_name()

# Group and sort by month number, then rename for display
trend = (customers
         .groupby(["signup_year", "signup_month_num", "signup_month"])["customer_id"]
         .count()
         .reset_index()
         .sort_values(["signup_year", "signup_month_num"])
         .pivot(index="signup_year", columns="signup_month", values="customer_id")
)

# Reorder columns by calendar order
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
trend = trend.reindex(columns=[m for m in month_order if m in trend.columns])

# Add row total and column total for business readability
trend["Total"] = trend.sum(axis=1)
trend.loc["Total"] = trend.sum()

trend

signup_month,January,February,March,April,May,June,July,August,September,October,November,December,Total
signup_year,,,,,,,,,,,,,
2022,NaN,NaN,NaN,NaN,NaN,NaN,1037.0,1048.0,1008.0,1018.0,1044.0,1045.0,6200.0
2023,1051.0,992.0,1049.0,990.0,1049.0,1051.0,1092.0,1039.0,1015.0,1006.0,1089.0,1036.0,12459.0
2024,1102.0,1012.0,1077.0,1002.0,1073.0,1050.0,25.0,NaN,NaN,NaN,NaN,NaN,6341.0
Total,2153.0,2004.0,2126.0,1992.0,2122.0,2101.0,2154.0,2087.0,2023.0,2024.0,2133.0,2081.0,25000.0


In [16]:
#Observations:

#1. 2022 data starts from July — partial year, not comparable full-year
#2. 2024 data ends at July — dataset likely extracted mid-2024
#3. Only 2023 is a complete year — use as primary baseline
#4. For YoY comparisons, use Jul-Dec window (common across 2022 & 2023)
#5. 2024 July shows only 25 signups — likely data cutoff mid-month

In [17]:
# Jan-June: Common between 2023 and 2024
h1_months = ["January", "February", "March", "April", "May", "June"]

# Jul-Dec: Common between 2022 and 2023
h2_months = ["July", "August", "September", "October", "November", "December"]

h1 = trend[h1_months].loc[[2023,2024]]
h1["h1_total"] = h1.sum(axis=1)
h1_growth = round(h1["h1_total"].pct_change()*100,2)



h2 = trend[h2_months].loc[[2022,2023]]
h2["h2_total"] = h2.sum(axis=1)
h2_growth = round(h2["h2_total"].pct_change()*100,2)

In [18]:
print(h1_growth.round(2).astype(str) + "%")


signup_year
2023      NaN
2024    2.17%
Name: h1_total, dtype: str


In [19]:
print(h2_growth.round(2).astype(str) + "%")


signup_year
2022      NaN
2023    1.24%
Name: h2_total, dtype: str


In [20]:

# 2. What is the segment distribution across cities?
# Which cities have the most Premium customers vs Budget? 
# Helps identify high-value markets
customers_2023 =  customers[customers["signup_year"]==2023]

seg_piv = customers_2023.groupby(["segment","city"])["customer_id"].count().reset_index().pivot(index='city',columns="segment",values="customer_id")

seg_piv.loc[:,"Total"] = seg_piv.sum(axis =1)
seg_piv.loc["Total",:] = seg_piv.sum(axis =0)

# Premium % share per city — most useful for business
seg_piv["Premium %"] = ((seg_piv["Premium"] / seg_piv["Total"]) * 100).round(1)


In [21]:
seg_piv.iloc[:-1,:].sort_values(by = "Premium %",ascending=False)

segment,Budget,Premium,Value,Total,Premium %
city,,,,,
Chandigarh,141.0,79.0,149.0,369.0,21.4
Bangalore,704.0,354.0,641.0,1699.0,20.8
Jaipur,233.0,110.0,188.0,531.0,20.7
Hyderabad,514.0,260.0,516.0,1290.0,20.2
Lucknow,194.0,100.0,204.0,498.0,20.1
Delhi,787.0,382.0,788.0,1957.0,19.5
Mumbai,912.0,442.0,920.0,2274.0,19.4
Kochi,122.0,53.0,100.0,275.0,19.3
Pune,413.0,193.0,396.0,1002.0,19.3


In [22]:
#5. What is the signup growth rate year over year per segment?
# Are Premium customers growing faster than Budget? 
# Tells us if we are moving upmarket or downmarket

# H1 (Jan-Jun): Compare 2023 vs 2024
h1_seg = (customers[customers["signup_month"].isin(["January","February","March","April","May","June"])]
          .groupby(["signup_year","segment"])["customer_id"]
          .count()
          .unstack(fill_value=0)
          .loc[[2023,2024]])

h1_seg.loc["Total"] = h1_seg.sum()
h1_seg.loc["Growth %"] = ((h1_seg.loc[2024] - h1_seg.loc[2023]) / h1_seg.loc[2023] * 100).round(2)

print("H1 (Jan-Jun) Segment Growth: 2023 vs 2024")
print(h1_seg)

# H2 (Jul-Dec): Compare 2022 vs 2023
h2_seg = (customers[customers["signup_month"].isin(["July","August","September","October","November","December"])]
          .groupby(["signup_year","segment"])["customer_id"]
          .count()
          .unstack(fill_value=0)
          .loc[[2022,2023]])

h2_seg.loc["Total"] = h2_seg.sum()
h2_seg.loc["Growth %"] = ((h2_seg.loc[2023] - h2_seg.loc[2022]) / h2_seg.loc[2022] * 100).round(2)

print("\nH2 (Jul-Dec) Segment Growth: 2022 vs 2023")
print(h2_seg)

H1 (Jan-Jun) Segment Growth: 2023 vs 2024
segment       Budget  Premium    Value
signup_year                           
2023         2511.00  1238.00  2433.00
2024         2573.00  1230.00  2513.00
Total        5084.00  2468.00  4946.00
Growth %        2.47    -0.65     3.29

H2 (Jul-Dec) Segment Growth: 2022 vs 2023
segment       Budget  Premium    Value
signup_year                           
2022         2518.00  1215.00  2467.00
2023         2555.00  1222.00  2500.00
Total        5073.00  2437.00  4967.00
Growth %        1.47     0.58     1.34


In [25]:
customers = customers.drop("signup_month_num",axis=1)

In [28]:
customers.to_parquet("..\Data\preprocessed\customers_cleaned.parquet")

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\naban\AppData\Local\Temp\ipykernel_11788\2416907174.py:1: SyntaxWarning: invalid escape sequence '\D'
  customers.to_parquet("..\Data\preprocessed\customers_cleaned.parquet")
